# 深度神经网络应用：图像分类

当你完成此 Notebook 时，你将完成第 4 周的最后一个编程作业，以及课程 1 的最后一个编程作业！走吧你！ 

要构建猫/非猫分类器，你将使用之前的作业中的函数来构建深度网络。希望你会发现比之前的逻辑回归实现的准确性有所提高。  

**在此作业之后，你将能够：**

- 构建并训练深度L层神经网络，并将其应用于监督学习

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/iLwon/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 加载和处理数据集](#2)
- [3 - 模型架构](#3)
    - [3.1 - 2 层神经网络](#3-1)
    - [3.2 - L层深度神经网络](#3-2)
    - [3.3 - 一般方法](#3-3)
- [4 - 两层神经网络](#4)
    - [练习 1 - two_layer_model](#ex-1)
    - [4.1 - 训练模型](#4-1)
- [5 - L 层神经网络](#5)
    - [练习 2 - L_layer_model](#ex-2)
    - [5.1 - 训练模型](#5-1)
- [6 - 结果分析](#6)
- [7 - 使用你自己的图像进行测试（可选/不计分的练习）](#7)

<a name='1'></a>
## 1 - 依赖库

首先导入作业期间所需的所有包。 

- [NumPy](https://www.numpy.org/) 是使用 Python 进行科学计算的基础包。
- [matplotlib](http://matplotlib.org) 是一个用 Python 绘制图形的库。
- [h5py](http://www.h5py.org) 是一个与存储在 H5 文件中的数据集交互的通用包。
- [太平](http://www.pythonware.com/products/pil/) 和 [scipy](https://www.scipy.org/) 在这里用来测试你的模型，最后用你自己的图片来测试。
- `dnn_app_utils` 将“构建深度神经网络：一步一步”作业中实现的函数提供给此 Notebook。
- `np.random.seed(1)` 用于保持所有随机函数调用的一致性。它有助于对你的作业进行评分 - 所以请不要更改它！ 

In [ ]:
### v1.2

In [ ]:
import time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy
from PIL import Image
from scipy import ndimage
from dnn_app_utils_v3 import *
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - 加载和处理数据集

你将使用与“神经网络的逻辑回归”(作业 2) 中相同的“猫与非猫”数据集。你当时构建的模型在分类猫与非猫图像方面的测试准确率为 70%。希望你的新模型能够表现得更好！

**问题陈述**：给你一个数据集（“data.h5”），其中包含：
    - 训练集的 `m_train` 图像标记为猫 (1) 或非猫 (0)
    - 测试集的 `m_test` 图像标记为猫和非猫
    - 每个图像的形状为 (num_px, num_px, 3)，其中 3 表示 3 个通道 (RGB)。

让我们更熟悉一下数据集。通过运行下面的单元格加载数据。

In [ ]:
train_x_orig, train_y, test_x_orig, test_y, classes = load_data()

以下代码将向你显示数据集中的图像。请随意更改索引并多次重新运行单元以检查其他图像。 

In [ ]:
# 图片示例
index = 10
plt.imshow(train_x_orig[index])
print ("y = " + str(train_y[0,index]) + ". It's a " + classes[train_y[0,index]].decode("utf-8") +  " picture.")

In [ ]:
# 探索你的数据集 
m_train = train_x_orig.shape[0]
num_px = train_x_orig.shape[1]
m_test = test_x_orig.shape[0]

print ("Number of training examples: " + str(m_train))
print ("Number of testing examples: " + str(m_test))
print ("Each image is of size: (" + str(num_px) + ", " + str(num_px) + ", 3)")
print ("train_x_orig shape: " + str(train_x_orig.shape))
print ("train_y shape: " + str(train_y.shape))
print ("test_x_orig shape: " + str(test_x_orig.shape))
print ("test_y shape: " + str(test_y.shape))

像往常一样，你在将图像输入网络之前对其进行整形和标准化。代码在下面的单元格中给出。

<img src="images/imvectorkiank.png" style="width:450px;height:300px;">
<div style="text-align:center;color:purple;"><b>图1</b>：图像到向量的转换。</div>

In [ ]:
# 重塑训练和测试示例 
train_x_flatten = train_x_orig.reshape(train_x_orig.shape[0], -1).T   # "-1" 使重塑压平剩余尺寸
test_x_flatten = test_x_orig.reshape(test_x_orig.shape[0], -1).T

# 标准化数据，使其特征值介于 0 和 1 之间。
train_x = train_x_flatten/255.
test_x = test_x_flatten/255.

print ("train_x's shape: " + str(train_x.shape))
print ("test_x's shape: " + str(test_x.shape))

**注意**：
$12,288$ 等于 $64 \times 64 \times 3$，这是一个重塑图像向量的大小。

<a name='3'></a>
## 3 - 模型架构

<a name='3-1'></a>
### 3.1 - 2 层神经网络

现在你已经熟悉了数据集，是时候构建一个深度神经网络来区分猫图像和非猫图像了！

你将构建两个不同的模型：

- 2 层神经网络
- L 层深度神经网络

然后，你将比较这些模型的性能，并尝试 $L$ 的一些不同值。 

我们来看看这两种架构：

<img src="images/2layerNN_kiank.png" style="width:650px;height:400px;">
<div style="text-align:center;color:purple;"><b>图2</b>：2层神经网络。 <br> 模型可概括为： INPUT -> LINEAR -> ReLU -> LINEAR -> sigmoid -> OUTPUT.</div>

<u><b>图2</b></u>的详细架构：
- 输入是一个 (64,64,3) 图像，它被展平为大小为 $(12288,1)$ 的向量。 
- 相应的向量：$[x_0,x_1,...,x_{12287}]^T$ 然后乘以大小为 $(n^{[1]}, 12288)$ 的权重矩阵 $W^{[1]}$。
- 然后，添加一个偏置项并取其ReLU，得到以下向量：$[a_0^{[1]}, a_1^{[1]},..., a_{n^{[1]}-1}^{[1]}]^T$。
- 将所得向量乘以 $W^{[2]}$ 并加上截距（偏置）。 
- 最后取结果的sigmoid。如果大于 0.5，则将其分类为猫。

<a name='3-2'></a>
### 3.2 - L层深度神经网络

使用上述表示来表示 L 层深度神经网络是相当困难的。然而，这是一个简化的网络表示：

<img src="images/LlayerNN_kiank.png" style="width:650px;height:400px;">
<div style="text-align:center;color:purple;">

<b>图3</b>：L层神经网络。 <br> 模型可概括为： [LINEAR -> ReLU] $\times$ (L-1) -> LINEAR -> SIGMOID

</div>

<u><b>图3</b></u>的详细架构：
- 输入是 (64,64,3) 图像，该图像被展平为大小为 (12288,1) 的向量。
- 相应的向量：$[x_0,x_1,...,x_{12287}]^T$ 然后乘以权重矩阵 $W^{[1]}$，然后添加截距 $b^{[1]}$。结果称为线性单位。
- 接下来，取线性单元的ReLU。根据模型架构，可以对每个 $(W^{[l]}, b^{[l]})$ 重复此过程多次。
- 最后取最终线性单元的sigmoid。如果大于0.5，则将其分类为猫。

<a name='3-3'></a>
### 3.3 - 一般方法

像往常一样，你将遵循深度学习方法来构建模型：

1. 初始化参数/定义超参数
2. num_iterations 的循环：
    a. 前向传播
    b．计算代价函数
    c. 反向传播
    d．更新参数（使用参数和反向传播的梯度） 
3. 使用经过训练的参数来预测标签

现在继续实现这两个模型！

<a name='4'></a>
## 4 - 两层神经网络

<a name='ex-1'></a>
### 练习 1 - two_layer_model 

使用你在之前的作业中实现的辅助函数构建一个具有以下结构的 2 层神经网络：*LINEAR -> ReLU -> LINEAR -> sigmoid*。 函数及其输入是：
```python
def initialize_parameters(n_x, n_h, n_y):
    ...
    return parameters 
def linear_activation_forward(A_prev, W, b, activation):
    ...
    return A, cache
def compute_cost(AL, Y):
    ...
    return cost
def linear_activation_backward(dA, cache, activation):
    ...
    return dA_prev, dW, db
def update_parameters(parameters, grads, learning_rate):
    ...
    return parameters
```

In [ ]:
### 定义模型的常数####
n_x = 12288     # num_px * num_px * 3
n_h = 7
n_y = 1
layers_dims = (n_x, n_h, n_y)
learning_rate = 0.0075

In [ ]:
# GRADED FUNCTION: two_layer_model

def two_layer_model(X, Y, layers_dims, learning_rate = 0.0075, num_iterations = 3000, print_cost=False):
    """
    实现两层神经网络：LINEAR->ReLU->LINEAR->sigmoid。
    
    参数：
    X -- 输入数据，形状（n_x，示例数量）
    Y -- true "label" 向量（如果是猫则包含 1，如果非猫则包含 0），形状（1，示例数）
    layers_dims -- 层的尺寸 (n_x, n_h, n_y)
    num_iterations -- 优化循环的迭代次数
    learning_rate -- 梯度下降更新规则的学习率
    print_cost -- 如果设置为 True，这将打印每 100 次迭代的代价 
    
    返回：
    parameters -- 包含W1、W2、b1和b2的字典
    """
    
    np.random.seed(1)
    grads = {}
    costs = []                              # 跟踪代价
    m = X.shape[1]                           # 示例数量
    (n_x, n_h, n_y) = layers_dims
    
    # 通过调用你之前实现的函数之一来初始化参数字典
    #（≈ 1 行代码）
    # parameters = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 从字典参数中获取W1、b1、W2和b2。
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # 循环 (梯度下降)

    for i in range(0, num_iterations):

        # 前向传播：线性 -> ReLU -> 线性 -> sigmoid。输入："X, W1, b1, W2, b2"。输出："A1, cache1, A2, cache2"。
        #（约2行代码）
        # A1, cache1 = ...
        # A2, cache2 = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 计算代价
        #（≈ 1 行代码）
        # cost = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 初始化反向传播
        dA2 = - (np.divide(Y, A2) - np.divide(1 - Y, 1 - A2))
        
        # 反向传播。输入："dA2, cache2, cache1"。输出：dA1、dW2、db2，以及 dA0（此处不使用）、dW1、db1。
        #（约2行代码）
        # dA1, dW2, db2 = ...
        # dA0, dW1, db1 = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 将梯度['dWl']设置为dW1，梯度['db1']设置为db1，梯度['dW2']设置为dW2，梯度['db2']设置为db2
        grads['dW1'] = dW1
        grads['db1'] = db1
        grads['dW2'] = dW2
        grads['db2'] = db2
        
        # 更新参数。
        #（约1行代码）
        # parameters = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

        # 从参数中检索 W1, b1, W2, b2
        W1 = parameters["W1"]
        b1 = parameters["b1"]
        W2 = parameters["W2"]
        b2 = parameters["b2"]
        
        # 打印每 100 次迭代和最后一次迭代的代价
        if print_cost and (i % 100 == 0 or i == num_iterations - 1):
            print("Cost after iteration {}: {}".format(i, np.squeeze(cost)))
        if i % 100 == 0:
            costs.append(cost)
            
    return parameters, costs

def plot_costs(costs, learning_rate=0.0075):
    plt.plot(np.squeeze(costs))
    plt.ylabel('cost')
    plt.xlabel('iterations (per hundreds)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2, print_cost=False)

print("Cost after first iteration: " + str(costs[0]))

two_layer_model_test(two_layer_model)

**预期输出：**

```
cost after iteration 1 must be around 0.69
```

<a name='4-1'></a>
### 4.1 - 训练模型 

如果你的代码通过了前一个单元，请运行下面的单元来训练你的参数。 

- 每次迭代的代价都应该降低。 

- 运行 2500 次迭代最多可能需要 5 分钟。 

In [ ]:
parameters, costs = two_layer_model(train_x, train_y, layers_dims = (n_x, n_h, n_y), num_iterations = 2500, print_cost=True)
plot_costs(costs, learning_rate)

**预期输出**：
<table> 
    <tr>
        <td> <b> 迭代后代价 0</b></td>
        <td> 0.6930497356599888 </td>
    </tr>
    <tr>
        <td> <b>迭代后代价 100</b></td>
        <td> 0.6464320953428849 </td>
    </tr>
    <tr>
        <td> <b>...</b></td>
        <td> ... </td>
    </tr>
    <tr>
        <td> <b>迭代后代价 2499</b></td>
        <td> 0.04421498215868956 </td>
    </tr>
</table>

**很好！** 你成功训练了模型。幸好你构建了向量化的实现！否则，训练它可能需要花费 10 倍的时间。

现在，你可以使用经过训练的参数对数据集中的图像进行分类。要查看你对训练和测试集的预测，请运行下面的单元格。

In [ ]:
predictions_train = predict(train_x, train_y, parameters)

**预期输出**：
<table> 
    <tr>
        <td> <b>精度</b></td>
        <td> 0.9999999999999998 </td>
    </tr>
</table>

In [ ]:
predictions_test = predict(test_x, test_y, parameters)

**预期输出**：

<table> 
    <tr>
        <td> <b>精度</b></td>
        <td> 0.72 </td>
    </tr>
</table>

### 恭喜！看起来你的 2 层神经网络的性能 (72%) 比逻辑回归实现（70%，作业第 2 周）更好。让我们看看你是否可以使用 $L$ 层模型做得更好。

**注意**：你可能会注意到，在较少的迭代次数（例如 1500 次）上运行模型可以在测试集上提供更好的准确性。这称为“提前停止”，你将在下一课程中听到更多相关内容。提前停止是防止过拟合的一种方法。 

<a name='5'></a>
## 5 - L 层神经网络

<a name='ex-2'></a>
### 练习 2 - L_layer_model 

使用之前实现的帮助器函数构建一个 $L$ 层神经网络，其结构如下：*[LINEAR -> ReLU]$\times$(L-1) -> LINEAR -> sigmoid*。 函数及其输入是：
```python
def initialize_parameters_deep(layers_dims):
    ...
    return parameters 
def L_model_forward(X, parameters):
    ...
    return AL, caches
def compute_cost(AL, Y):
    ...
    return cost
def L_model_backward(AL, Y, caches):
    ...
    return grads
def update_parameters(parameters, grads, learning_rate):
    ...
    return parameters
```

In [ ]:
### 常数 ###
layers_dims = [12288, 20, 7, 5, 1] #  4层模型

In [ ]:
# GRADED FUNCTION: L_layer_model

def L_layer_model(X, Y, layers_dims, learning_rate = 0.0075, num_iterations = 3000, print_cost=False):
    """
    实现L层神经网络：[LINEAR->ReLU]*(L-1)->LINEAR->sigmoid。
    
    参数：
    X -- 输入数据，形状（n_x，示例数量）
    Y -- true "label" 向量（如果是猫则包含 1，如果非猫则包含 0），形状（1，示例数）
    layers_dims -- 包含输入大小和每层大小的列表，长度为（层数 + 1）。
    learning_rate -- 梯度下降更新规则的学习率
    num_iterations -- 优化循环的迭代次数
    print_cost -- 如果为 True，则每 100 步打印一次代价
    
    返回：
    parameters -- 模型学习到的参数。然后可以使用它们进行预测。
    """

    np.random.seed(1)
    costs = []                         # 跟踪代价
    
    # 参数初始化。
    #（≈ 1 行代码）
    # parameters = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 循环 (梯度下降)
    for i in range(0, num_iterations):

        # 前向传播：[线性 -> ReLU]*(L-1) -> 线性 -> sigmoid。
        #（≈ 1 行代码）
        # AL, caches = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        # 计算代价。
        #（≈ 1 行代码）
        # cost = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
        # 反向传播.
        #（≈ 1 行代码）
        # grads = ...    
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
 
        # 更新参数。
        #（≈ 1 行代码）
        # parameters = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
                
        # 打印每 100 次迭代和最后一次迭代的代价
        if print_cost and (i % 100 == 0 or i == num_iterations - 1):
            print("Cost after iteration {}: {}".format(i, np.squeeze(cost)))
        if i % 100 == 0:
            costs.append(cost)
    
    return parameters, costs

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 1, print_cost = False)

print("Cost after first iteration: " + str(costs[0]))

L_layer_model_test(L_layer_model)

**预期输出：**

```
cost after iteration 1 must be around 0.77
```

<a name='5-1'></a>
### 5.1 - 训练模型 

如果你的代码通过了前一个单元，请运行下面的单元将你的模型训练为 4 层神经网络。 

- 每次迭代的代价都应该降低。 

- 运行 2500 次迭代最多可能需要 5 分钟。 

In [ ]:
parameters, costs = L_layer_model(train_x, train_y, layers_dims, num_iterations = 2500, print_cost = True)

**预期输出**：
<table> 
    <tr>
        <td> <b> 迭代后代价 0</b></td>
        <td> 0.771749 </td>
    </tr>
    <tr>
        <td> <b>迭代后代价 100</b></td>
        <td> 0.672053 </td>
    </tr>
    <tr>
        <td> <b>...</b></td>
        <td> ... </td>
    </tr>
    <tr>
        <td> <b>迭代后代价 2499</b></td>
        <td> 0.088439 </td>
    </tr>
</table>

In [ ]:
pred_train = predict(train_x, train_y, parameters)

**预期输出**：

<table>
    <tr>
    <td>
        <b>训练精度</b>
    </td>
    <td>
    0.985645933014
    </td>
    </tr>
</table>

In [ ]:
pred_test = predict(test_x, test_y, parameters)

**预期输出**：

<table> 
    <tr>
        <td> <b>测试精度</b></td>
        <td> 0.8 </td>
    </tr>
</table>

### 恭喜！看起来你的 4 层神经网络在相同的测试集上比你的 2 层神经网络 (72%) 具有更好的性能 (80%)。 

对于这项任务来说，这已经是相当不错的表现了。干得好！ 

在下一门“改进深度神经网络”课程中，你将能够通过系统地搜索更好的超参数来获得更高的准确率：例如，learning_rate、layers_dims 或 num_iterations。  

<a name='6'></a>
##  6 - 结果分析

首先，看一下 L 层模型标记错误的一些图像。这将显示一些标签错误的图像。 

In [ ]:
print_mislabeled_images(classes, test_x, test_y, pred_test)

**模型在一些类型的图像上表现不佳，包括：** 
- 猫的身体处于不寻常的位置
- 猫出现在相似颜色的背景下
- 不寻常的猫颜色和种类
- 拍摄角度
- 图片亮度
- 尺度变化（猫在图像中很大或很小） 

### 恭喜你完成了这个作业！ 

你刚刚构建并训练了一个深度 L 层神经网络，并应用它来区分猫和非猫，这是深度学习中非常严肃且重要的任务。 ;) 

至此，你已完成深度学习专项课程中第 1 门课程的全部作业！下面还有一道不计分的选做练习，可以用自己的图像测试模型，看看它是否会将你识别成猫。

干得好，希望在下一门课程中见到你！ 

<a name='7'></a>
## 7 - 使用你自己的图像进行测试（可选/不计分的练习） ##

从现在起，如果你愿意，你可以使用自己的图像来测试模型的输出。为此，请按照下列步骤操作：

1. 单击此 Notebook 上部栏中的“文件”，然后单击“打开”以转到你的 Coursera Hub。
2. 将图像添加到 Jupyter Notebook 目录的“images”文件夹中
3. 在以下代码中更改图像的名称
4. 运行代码并检查算法是否正确（1 = 猫，0 = 非猫）！

In [ ]:
## START CODE HERE ##
my_image = "my_image.jpg" # 将其更改为图像文件的名称 
my_label_y = [1] # 图像的真实类别（1 -> 猫，0 -> 非猫）
## END CODE HERE ##

fname = "images/" + my_image
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T

my_predicted_image = predict(image, my_label_y, parameters)


print ("y = " + str(np.squeeze(my_predicted_image)) + ", your L-layer model predicts a \"" + classes[int(np.squeeze(my_predicted_image)),].decode("utf-8") +  "\" picture.")

**参考文献**：

- 自动加载外部模块：http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython